# Data Exploration
Exploratory analysis of the four raw datasets:
- `posek.csv` — ZGS logging records
- `vreme.csv` — ARSO weather station records
- `odseki_gozdno.gpkg` — forest sections (spatial)
- `sestoji.gpkg` — forest stands (spatial)

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 30)

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'

---
## 1. posek.csv — Logging Records

In [ ]:
posek = pd.read_csv(RAW / 'ZGS' / 'posek.csv')
posek.columns = posek.columns.str.strip()
posek['odsek'] = posek['odsek'].str.strip()
posek['posekano'] = pd.to_datetime(posek['posekano'], errors='coerce')

print(f'Shape: {posek.shape}')
posek.head()

In [ ]:
posek.dtypes

In [ ]:
posek.describe()

In [ ]:
print('Missing values:')
posek.isnull().sum()

In [ ]:
print(f'Date range: {posek["posekano"].min()} → {posek["posekano"].max()}')
print(f'Unique odsek: {posek["odsek"].nunique():,}')
print(f'Unique GGO:   {posek["ggo"].nunique()}')
print(f'Unique vrsec: {posek["vrsec"].nunique()} → {sorted(posek["vrsec"].unique())}')

In [ ]:
# Annual logged volume
posek['leto'] = posek['posekano'].dt.year
annual = posek.groupby('leto')['kubikov'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(annual['leto'], annual['kubikov'] / 1e6, color='steelblue')
ax.set_xlabel('Year')
ax.set_ylabel('Volume (million m³)')
ax.set_title('Annual Logged Volume (posek)')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of kubikov (log scale)
fig, ax = plt.subplots(figsize=(8, 4))
posek['kubikov'].clip(upper=posek['kubikov'].quantile(0.99)).hist(bins=80, ax=ax, color='steelblue', edgecolor='none')
ax.set_xlabel('Kubikov (m³)  [clipped at 99th pct]')
ax.set_ylabel('Count')
ax.set_title('Distribution of logged volume per record')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 odsek by total volume
posek.groupby('odsek')['kubikov'].sum().nlargest(10).sort_values().plot(
    kind='barh', figsize=(8, 4), color='steelblue', title='Top 10 odsek by total logged volume (m³)'
)
plt.xlabel('m³')
plt.tight_layout()
plt.show()

---
## 2. vreme.csv — Weather Records

In [ ]:
vreme = pd.read_csv(
    RAW / 'ARSO' / 'vreme.csv',
    encoding='utf-8',
    low_memory=False,
)
# Fix double-encoded column names → clean Slovenian labels
vreme.columns = [
    'station_id', 'datum',
    'temp_avg_C', 'temp_max_C', 'temp_min_C',
    'padavine_mm', 'snezna_odeja_cm', 'novi_sneg_cm',
    'nevihta', 'toca', 'viharni_veter',
]
vreme['datum'] = pd.to_datetime(vreme['datum'], errors='coerce')

print(f'Shape: {vreme.shape}')
vreme.head()

In [ ]:
vreme.dtypes

In [ ]:
vreme.describe()

In [ ]:
print('Missing values (%):')
(vreme.isnull().sum() / len(vreme) * 100).round(1)

In [ ]:
print(f'Date range:      {vreme["datum"].min()} → {vreme["datum"].max()}')
print(f'Unique stations: {vreme["station_id"].nunique()}')
print(f'Station IDs:     {sorted(vreme["station_id"].unique())}')

In [ ]:
# Records per station
vreme['station_id'].value_counts().sort_index().plot(
    kind='bar', figsize=(12, 4), color='steelblue',
    title='Number of daily records per weather station'
)
plt.xlabel('Station ID')
plt.ylabel('Records')
plt.tight_layout()
plt.show()

In [ ]:
# Temperature distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, col, label in zip(axes,
    ['temp_avg_C', 'temp_max_C', 'temp_min_C'],
    ['Avg temp (°C)', 'Max temp (°C)', 'Min temp (°C)']):
    vreme[col].dropna().hist(bins=60, ax=ax, color='tomato', edgecolor='none')
    ax.set_title(label)
    ax.set_ylabel('Count')
plt.suptitle('Temperature distributions (all stations, all years)')
plt.tight_layout()
plt.show()

In [ ]:
# Precipitation distribution (clipped)
fig, ax = plt.subplots(figsize=(8, 4))
vreme['padavine_mm'].clip(upper=vreme['padavine_mm'].quantile(0.99)).dropna().hist(
    bins=60, ax=ax, color='royalblue', edgecolor='none'
)
ax.set_xlabel('Precipitation (mm)  [clipped at 99th pct]')
ax.set_ylabel('Count')
ax.set_title('Daily precipitation distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Annual average temperature per station
vreme['leto'] = vreme['datum'].dt.year
annual_temp = vreme.groupby(['leto', 'station_id'])['temp_avg_C'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for sid, grp in annual_temp.groupby('station_id'):
    ax.plot(grp['leto'], grp['temp_avg_C'], marker='', linewidth=1, label=str(sid), alpha=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Avg temperature (°C)')
ax.set_title('Annual mean temperature per station')
ax.legend(title='Station', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## 3. odseki_gozdno.gpkg — Forest Sections

In [ ]:
odseki = gpd.read_file(RAW / 'ZGS' / 'odseki_gozdno.gpkg')

print(f'Shape:   {odseki.shape}')
print(f'CRS:     {odseki.crs}')
print(f'GGO units: {odseki["ggo"].nunique()}')
odseki.drop(columns='geometry').head()

In [ ]:
odseki.drop(columns='geometry').describe(include='all').T

In [ ]:
print('Missing values (%):')
(odseki.drop(columns='geometry').isnull().sum() / len(odseki) * 100).round(1)[lambda s: s > 0]

In [ ]:
# Area distribution per section
fig, ax = plt.subplots(figsize=(8, 4))
odseki['povrsina'].clip(upper=odseki['povrsina'].quantile(0.99)).hist(
    bins=60, ax=ax, color='forestgreen', edgecolor='none'
)
ax.set_xlabel('Area (ha)  [clipped at 99th pct]')
ax.set_ylabel('Count')
ax.set_title('Distribution of forest section areas')
plt.tight_layout()
plt.show()

In [ ]:
# Total area per GGO
odseki.groupby('ggo_naziv')['povrsina'].sum().sort_values(ascending=True).plot(
    kind='barh', figsize=(8, 8), color='forestgreen',
    title='Total forest area per GGO (ha)'
)
plt.xlabel('Area (ha)')
plt.tight_layout()
plt.show()

In [ ]:
# Spatial map coloured by GGO
fig, ax = plt.subplots(figsize=(14, 10))
odseki.plot(column='ggo', ax=ax, legend=True, cmap='tab20',
            legend_kwds={'bbox_to_anchor': (1.01, 1), 'loc': 'upper left', 'fontsize': 8})
ax.set_title('Forest sections coloured by GGO', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---
## 4. sestoji.gpkg — Forest Stands

In [ ]:
sestoji = gpd.read_file(RAW / 'ZGS' / 'sestoji.gpkg')

print(f'Shape: {sestoji.shape}')
print(f'CRS:   {sestoji.crs}')
sestoji.drop(columns='geometry').head()

In [ ]:
sestoji.drop(columns='geometry').describe().T

In [ ]:
print('Missing values (%):')
(sestoji.drop(columns='geometry').isnull().sum() / len(sestoji) * 100).round(1)[lambda s: s > 0]

In [ ]:
# Development phase (rfaza) distribution
sestoji['rfaza_naziv'].value_counts().sort_values().plot(
    kind='barh', figsize=(8, 6), color='darkorange',
    title='Stand development phase (rfaza) distribution'
)
plt.xlabel('Number of stands')
plt.tight_layout()
plt.show()

In [ ]:
# Tree species composition: conifer vs broadleaf (lzskdv columns = basal area shares)
# lzigl = conifer growing stock (m³/ha), lzlst = broadleaf, lzsku = total
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label, color in zip(
    axes,
    ['lzigl', 'lzlst', 'lzsku'],
    ['Conifer (m³/ha)', 'Broadleaf (m³/ha)', 'Total growing stock (m³/ha)'],
    ['darkgreen', 'goldenrod', 'steelblue']
):
    sestoji[col].clip(upper=sestoji[col].quantile(0.99)).hist(
        bins=60, ax=ax, color=color, edgecolor='none'
    )
    ax.set_title(label)
    ax.set_ylabel('Count')
plt.suptitle('Growing stock distributions (clipped at 99th pct)')
plt.tight_layout()
plt.show()

In [ ]:
# Conifer share per stand
sestoji['del_iglavcev'] = sestoji['lzigl'] / sestoji['lzsku'].replace(0, float('nan'))
fig, ax = plt.subplots(figsize=(8, 4))
sestoji['del_iglavcev'].dropna().hist(bins=50, ax=ax, color='darkgreen', edgecolor='none')
ax.set_xlabel('Conifer share (0–1)')
ax.set_ylabel('Count')
ax.set_title('Conifer share distribution across stands')
plt.tight_layout()
plt.show()

In [ ]:
# Spatial map coloured by conifer share
fig, ax = plt.subplots(figsize=(14, 10))
sestoji.plot(column='del_iglavcev', ax=ax, cmap='RdYlGn_r',
             legend=True, missing_kwds={'color': 'lightgrey'},
             legend_kwds={'label': 'Conifer share', 'shrink': 0.6})
ax.set_title('Conifer share per forest stand', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---
## 5. Quick Cross-Dataset Summary

In [ ]:
summary = pd.DataFrame([
    {'dataset': 'posek.csv',           'rows': len(posek),   'cols': posek.shape[1],   'spatial': False},
    {'dataset': 'vreme.csv',           'rows': len(vreme),   'cols': vreme.shape[1],   'spatial': False},
    {'dataset': 'odseki_gozdno.gpkg',  'rows': len(odseki),  'cols': odseki.shape[1],  'spatial': True},
    {'dataset': 'sestoji.gpkg',        'rows': len(sestoji), 'cols': sestoji.shape[1], 'spatial': True},
]).set_index('dataset')

summary

In [ ]:
# Shared join key: odsek — check overlap between posek and odseki_gozdno
posek_ids   = set(posek['odsek'].dropna().unique())
odseki_ids  = set(odseki['odsek'].dropna().unique())
sestoji_ids = set(sestoji['odsek'].dropna().unique())

print(f'posek odsek count:          {len(posek_ids):,}')
print(f'odseki_gozdno odsek count:  {len(odseki_ids):,}')
print(f'sestoji odsek count:        {len(sestoji_ids):,}')
print()
print(f'posek ∩ odseki_gozdno:      {len(posek_ids & odseki_ids):,}')
print(f'posek ∩ sestoji:            {len(posek_ids & sestoji_ids):,}')
print(f'odseki_gozdno ∩ sestoji:    {len(odseki_ids & sestoji_ids):,}')